# AERIS Part 1 - A* Heuristic Comparison

**CSE518 Artificial Intelligence - Monsoon 2026**

This notebook is the Part 1 comparison deliverable. It compares four heuristics on multiple deterministic maps:

1. H1: Zero heuristic
2. H2: Geometric cost lower bound
3. H3: Relaxed-network shortest-cost heuristic
4. H4: max(H2, H3)

Metrics: path cost, nodes expanded, runtime, admissibility and consistency.


In [ ]:
from pathlib import Path
import json, math, heapq, time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

DATA = Path("../data")


In [ ]:
def load_map(path):
    payload=json.loads(Path(path).read_text())
    G=nx.Graph()
    for n in payload["nodes"]:
        G.add_node(str(n["id"]),pos=(float(n["x"]),float(n["y"])))
    for e in payload["edges"]:
        a,b=str(e["source"]),str(e["target"])
        is_open=bool(e.get("open",True))
        G.add_edge(a,b,cost=float(e["cost"]) if is_open else 1e6,known_open=is_open)
    goal=str(payload["requests"][0]["destination"])
    return G, "Depot", goal

def reconstruct(parent,goal):
    if goal not in parent: return []
    p=[];cur=goal
    while cur is not None:
        p.append(cur);cur=parent[cur]
    return p[::-1]

def h_zero(G,goal):
    return {n:0.0 for n in G.nodes}

def h_geo(G,goal):
    ratios=[]
    for a,b,d in G.edges(data=True):
        if d["cost"]>=1e5: continue
        dist=max(math.dist(G.nodes[a]["pos"],G.nodes[b]["pos"]),1e-9)
        ratios.append(d["cost"]/dist)
    k=min(ratios) if ratios else 0.0
    return {n:math.dist(G.nodes[n]["pos"],G.nodes[goal]["pos"])*k for n in G.nodes}

def h_relaxed(G,goal):
    H=nx.Graph();H.add_nodes_from(G.nodes(data=True))
    for a,b,d in G.edges(data=True):
        if d["cost"]>=1e5: continue
        eu=math.dist(G.nodes[a]["pos"],G.nodes[b]["pos"])
        H.add_edge(a,b,cost=min(float(d["cost"]),eu))
    out={}
    for n in G.nodes:
        try: out[n]=nx.shortest_path_length(H,n,goal,weight="cost")
        except: out[n]=0.0
    return out

def diagnostics(G,goal,h):
    exact={}
    for n in G.nodes:
        try: exact[n]=nx.shortest_path_length(G,n,goal,weight="cost")
        except: exact[n]=float("inf")
    admissible=all(h[n]<=exact[n]+1e-9 for n in G.nodes if np.isfinite(exact[n]))
    consistent=True
    for a,b,d in G.edges(data=True):
        if d["cost"]>=1e5: continue
        c=float(d["cost"])
        if h[a]>c+h[b]+1e-9 or h[b]>c+h[a]+1e-9:
            consistent=False
    return admissible,consistent

def astar(G,start,goal,h):
    pq=[(h[start],0,start)];counter=0;g={start:0.0};parent={start:None};closed=set();expanded=[]
    while pq:
        _,_,n=heapq.heappop(pq)
        if n in closed: continue
        closed.add(n);expanded.append(n)
        if n==goal: break
        for nb in G.neighbors(n):
            c=float(G[n][nb]["cost"])
            if c>=1e5: continue
            ng=g[n]+c
            if ng<g.get(nb,float("inf")):
                g[nb]=ng;parent[nb]=n;counter+=1
                heapq.heappush(pq,(ng+h[nb],counter,nb))
    path=reconstruct(parent,goal)
    cost=sum(G[a][b]["cost"] for a,b in zip(path,path[1:])) if path else float("inf")
    return path,cost,len(expanded)

def compare_map(path):
    G,start,goal=load_map(path)
    h1=h_zero(G,goal);h2=h_geo(G,goal);h3=h_relaxed(G,goal)
    h4={n:max(h2[n],h3[n]) for n in G.nodes}
    rows=[]
    for name,h in [("H1 Zero",h1),("H2 Geometric",h2),("H3 Relaxed",h3),("H4 max(H2,H3)",h4)]:
        t0=time.perf_counter()
        route,cost,expanded=astar(G,start,goal,h)
        dt=(time.perf_counter()-t0)*1000
        adm,cons=diagnostics(G,goal,h)
        rows.append({
            "Heuristic":name,"Route":" -> ".join(route),"Path cost":cost,
            "Expanded":expanded,"Runtime ms":dt,"Admissible":adm,"Consistent":cons
        })
    return pd.DataFrame(rows)


In [ ]:
all_results=[]
for path in sorted(DATA.glob("sample_map_*.json")):
    result=compare_map(path)
    result.insert(0,"Map",path.stem)
    all_results.append(result)

results=pd.concat(all_results,ignore_index=True)
results.round({"Path cost":3,"Runtime ms":4})


In [ ]:
summary = results.groupby("Heuristic").agg(
    mean_cost=("Path cost","mean"),
    mean_expanded=("Expanded","mean"),
    mean_runtime_ms=("Runtime ms","mean"),
    all_admissible=("Admissible","all"),
    all_consistent=("Consistent","all"),
).reset_index()
summary.round(4)


In [ ]:
pivot=results.pivot(index="Map",columns="Heuristic",values="Expanded")
ax=pivot.plot(kind="bar",figsize=(10,5))
ax.set_ylabel("Nodes expanded")
ax.set_title("A* heuristic comparison across deterministic maps")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Interpretation checklist

- H1 is the uninformed A* baseline and should behave like UCS.
- H2 introduces a cheap geometric lower bound.
- H3 is derived from a relaxed problem.
- H4 takes the maximum of two admissible heuristics, so it should be at least as informed as either component heuristic.
- The important comparison is **not only runtime**. We also verify admissibility, consistency, path cost, and search effort.
